# AI Code Auditor v4 — Live Demo
**Model:** DeepSeek-Coder-6.7B + QLoRA v4 | **Interface:** Gradio

### Before running:
1. GPU: T4 x1
2. Add dataset containing: `lora_adapter_v4/` folder
3. Run All — you will get a public URL at the end

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.45.3 gradio
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Environment set')

In [ ]:
import os, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'

# Find adapter path
ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'adapter_config.json' and 'checkpoint' not in root:
            ADAPTER_PATH = root

assert ADAPTER_PATH, 'adapter_config.json not found! Add lora_adapter_v4 dataset'
print('Adapter:', ADAPTER_PATH)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Load model
print('Loading model...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
base_model.config.use_cache = True

# Load v4 LoRA adapter
print('Loading v4 LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print('Model ready! VRAM:', round(torch.cuda.memory_allocated()/1e9, 1), 'GB')

In [ ]:
# Inference function
def analyze_code(code, use_rag=False):
    if not code.strip():
        return 'Please enter some code to analyze.', '', ''

    prompt = (
        'You are an expert security code auditor.\n'
        'Analyze the following C/C++ code and identify the security vulnerability.\n\n'
        f'```c\n{code}\n```\n\n'
        'Respond with the CWE type first, then explain and provide a secure rewrite.\n'
        'CWE:'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=400).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Extract CWE
    cwe_m = re.search(r'CWE-\d+', raw)
    cwe = cwe_m.group(0) if cwe_m else 'Unknown'

    # Extract explanation
    exp_m = re.search(r'Reason:\s*(.*?)(?:Fix:|```|$)', raw, re.DOTALL)
    explanation = exp_m.group(1).strip() if exp_m else raw[:300]

    # Extract secure code
    code_m = re.search(r'```(?:c|cpp)?\n(.*?)```', raw, re.DOTALL)
    secure = code_m.group(1).strip() if code_m else 'See full output below'

    return cwe, explanation, secure

# Quick test
test_code = 'int total = width * height * bpp; char *buf = malloc(total);'
cwe, exp, sec = analyze_code(test_code)
print('Test CWE:', cwe)
print('Model is working!')

In [ ]:
import gradio as gr

# Sample vulnerable code snippets for demo
EXAMPLES = [
    [
        'int allocate_image_buffer(int width, int height, int bpp) {\n'
        '    int stride = width * (bpp / 8);\n'
        '    int total = stride * height;\n'
        '    if (total > 0x1000000) return -1;\n'
        '    unsigned char *buf = malloc(total);\n'
        '    return buf ? 0 : -1;\n'
        '}'
    ],
    [
        'void handle_request(request_t *req) {\n'
        '    if (req->error) {\n'
        '        free(req);\n'
        '    }\n'
        '    log_request(req->id);\n'
        '    send_response(req->conn, 200);\n'
        '}'
    ],
    [
        'void copy_data(char *dest, char *src) {\n'
        '    char buffer[256];\n'
        '    strcpy(buffer, src);\n'
        '    strcpy(dest, buffer);\n'
        '}'
    ],
    [
        'int process_input(char *user_data, int len) {\n'
        '    if (len > MAX_SIZE) return -1;\n'
        '    char *buf = malloc(len);\n'
        '    memcpy(buf, user_data, len);\n'
        '    process(buf);\n'
        '    // forgot to free(buf)\n'
        '    return 0;\n'
        '}'
    ],
]

def gradio_analyze(code):
    if not code.strip():
        return '❌ Please enter code', '', ''
    cwe, explanation, secure_code = analyze_code(code)
    cwe_display = f'⚠️ VULNERABLE — {cwe}' if cwe != 'Unknown' else '❓ Unknown vulnerability'
    return cwe_display, explanation, secure_code

# Build Gradio interface
with gr.Blocks(title='AI Code Auditor v4') as demo:
    gr.Markdown("""
    # 🛡️ AI Code Auditor v4
    **Automated Security Vulnerability Detection** using DeepSeek-Coder-6.7B + QLoRA

    Trained on Big-Vul dataset (real CVEs) + 170 synthetic vulnerability samples.
    Detects: CWE-20, CWE-190, CWE-264, CWE-399, CWE-416 and more.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            code_input = gr.Code(
                label='📝 Paste C/C++ Code Here',
                language='c',
                lines=15
            )
            analyze_btn = gr.Button('🔍 Analyze Code', variant='primary', size='lg')
            gr.Examples(
                examples=EXAMPLES,
                inputs=code_input,
                label='📋 Example Vulnerable Code Snippets'
            )

        with gr.Column(scale=1):
            cwe_output = gr.Textbox(
                label='🎯 Vulnerability Classification',
                lines=2,
                interactive=False
            )
            explanation_output = gr.Textbox(
                label='📖 Explanation',
                lines=6,
                interactive=False
            )
            secure_output = gr.Code(
                label='🔒 Secure Rewrite',
                language='c',
                lines=10,
                interactive=False
            )

    analyze_btn.click(
        fn=gradio_analyze,
        inputs=code_input,
        outputs=[cwe_output, explanation_output, secure_output]
    )

    gr.Markdown("""
    ---
    **Model:** DeepSeek-Coder-6.7B | **Method:** QLoRA (4-bit NF4, LoRA r=16)
    **Dataset:** Big-Vul + 170 synthetic samples | **Accuracy:** 26% CWE, BLEU-4: 12.01
    """)

# Launch with public URL
print('Launching Gradio demo...')
demo.launch(
    share=True,
    debug=False,
    show_error=True
)